# Chapitre 2 — Leçon 3 : Le Pattern Fit/Predict

## Objectifs d'apprentissage

À la fin de cette leçon, vous serez capable de :
- **Utiliser** l'API universelle de scikit-learn : `.fit()`, `.predict()`, `.score()`
- **Distinguer** les paramètres (appris) des hyperparamètres (définis)
- **Appliquer** le pattern fit/transform pour le preprocessing

---

## 🎯 Accroche : Un langage universel

Imaginez que vous devez apprendre à utiliser 50 outils différents — chacun avec sa propre interface, ses propres commandes, sa propre logique. Ce serait épuisant !

Maintenant imaginez que tous ces outils partagent **exactement la même interface** : trois boutons qui font toujours la même chose.

C'est la philosophie de scikit-learn : **TOUS les modèles utilisent le même pattern**.

**Question :** Si vous apprenez à utiliser UN modèle scikit-learn, combien de modèles savez-vous utiliser ?

*(Réponse attendue : Tous ! Car ils partagent tous la même API)*

---

## 3.1 L'API universelle scikit-learn

### Les trois méthodes magiques

Voici le cœur de scikit-learn — trois méthodes que partagent TOUS les modèles :

```
┌─────────────────────────────────────────────────────────────────┐
│                 L'API UNIVERSELLE SCIKIT-LEARN                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌─────────────┐                                               │
│   │   .fit()    │  "Apprends à partir de ces données"          │
│   │             │  Entrée : X_train, y_train                    │
│   │             │  Le modèle ajuste ses paramètres internes     │
│   └─────────────┘                                               │
│          │                                                      │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ .predict()  │  "Fais des prédictions"                       │
│   │             │  Entrée : X_new (nouvelles données)           │
│   │             │  Sortie : y_pred (prédictions)                │
│   └─────────────┘                                               │
│          │                                                      │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  .score()   │  "Évalue ta performance"                      │
│   │             │  Entrée : X_test, y_test                      │
│   │             │  Sortie : score de performance                │
│   └─────────────┘                                               │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Voyons cela en action

Commençons par préparer un dataset simple pour nos expériences.

In [ ]:
# Importations nécessaires
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris, make_regression

# Créons un dataset de régression simple
X, y = make_regression(n_samples=200, n_features=3, noise=10, random_state=42)

# Séparation train/test (comme appris dans la leçon précédente !)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} exemples")
print(f"Test: {X_test.shape[0]} exemples")

---

## 3.2 `.fit()` : Entraîner le modèle

La méthode `.fit()` est le cœur de l'apprentissage. C'est là que la "magie" se produit : le modèle examine les données et **ajuste ses paramètres internes** pour capturer les patterns.

**Question :** Avant d'exécuter `.fit()`, le modèle sait-il faire des prédictions ?

*(Réponse attendue : Non ! Un modèle non entraîné n'a aucune connaissance des données)*

In [ ]:
# Créer un modèle (pas encore entraîné !)
model = LinearRegression()

# Le modèle ne sait rien pour l'instant
print("Modèle créé, mais pas encore entraîné.")
print(f"Type du modèle: {type(model).__name__}")

In [ ]:
# ENTRAÎNEMENT : le modèle apprend les patterns
model.fit(X_train, y_train)

print("✅ Modèle entraîné !")
print(f"\nCe que le modèle a appris :")
print(f"  - Coefficients (poids des features) : {model.coef_}")
print(f"  - Intercept (biais) : {model.intercept_:.2f}")

### 🔑 Réponse attendue

Après `.fit()`, le modèle a appris :
- **Les coefficients** : le poids (importance) de chaque feature
- **L'intercept** : la valeur de base quand toutes les features sont à 0

Ces valeurs sont appelées **paramètres** — elles sont **apprises** par le modèle à partir des données.

<details>
<summary>🤔 Question Socratique : Que se passe-t-il si vous appelez .fit() une deuxième fois avec des données différentes ?</summary>

### 🔑 Réponse

Le modèle **oublie** tout ce qu'il avait appris et **réapprend** à partir des nouvelles données. C'est un "reset" complet.

```python
model.fit(X_train_1, y_train_1)  # Apprend de dataset 1
model.fit(X_train_2, y_train_2)  # Oublie dataset 1, apprend de dataset 2
```

Si vous voulez combiner les connaissances de deux datasets, il faut les fusionner AVANT d'appeler `.fit()`.

</details>

---

## 3.3 `.predict()` : Faire des prédictions

Une fois le modèle entraîné, `.predict()` utilise les patterns appris pour prédire sur de **nouvelles données**.

In [ ]:
# PRÉDICTION sur les données de test
y_pred = model.predict(X_test)

print("Premières prédictions vs valeurs réelles :")
print("─" * 40)
for i in range(5):
    print(f"Prédit: {y_pred[i]:8.2f}  |  Réel: {y_test[i]:8.2f}  |  Erreur: {abs(y_pred[i] - y_test[i]):6.2f}")

**Question :** Pouvez-vous appeler `.predict()` sur un modèle qui n'a jamais été entraîné avec `.fit()` ?

*(Réponse attendue : Non ! Vous obtiendrez une erreur — le modèle n'a aucun paramètre à utiliser)*

In [ ]:
# Démonstration : essayer de prédire sans entraîner
model_non_entraine = LinearRegression()

try:
    # Ceci va échouer !
    predictions = model_non_entraine.predict(X_test)
except Exception as e:
    print(f"❌ Erreur attendue : {type(e).__name__}")
    print(f"   Message : {e}")

### Prédire sur une seule nouvelle observation

En production, vous recevrez souvent une seule nouvelle donnée à prédire.

In [ ]:
# Une nouvelle observation (3 features)
nouvelle_observation = [[0.5, -0.3, 1.2]]

# Prédiction
prediction = model.predict(nouvelle_observation)

print(f"Nouvelle observation : {nouvelle_observation[0]}")
print(f"Prédiction du modèle : {prediction[0]:.2f}")

⚠️ **Note importante** : `.predict()` attend un array 2D (même pour une seule observation). C'est pourquoi on utilise `[[...]]` avec des doubles crochets.

---

## 3.4 `.score()` : Évaluer la performance

La méthode `.score()` calcule automatiquement une métrique de performance.

| Type de modèle | Métrique par défaut de `.score()` |
|----------------|-----------------------------------|
| Régression | R² (coefficient de détermination) |
| Classification | Accuracy (taux de bons classements) |

In [ ]:
# Score sur les données de TEST
score_test = model.score(X_test, y_test)

# Score sur les données de TRAIN (pour comparaison)
score_train = model.score(X_train, y_train)

print(f"Performance du modèle :")
print(f"  - Score sur TRAIN : {score_train:.4f} (R²)")
print(f"  - Score sur TEST  : {score_test:.4f} (R²)")
print(f"\n📊 Interprétation : Le modèle explique {score_test*100:.1f}% de la variance sur les données de test.")

**Question :** Pourquoi est-il important de regarder le score sur TRAIN et sur TEST ?

*(Réponse attendue : Pour détecter l'overfitting — si train >> test, le modèle mémorise au lieu de généraliser)*

<details>
<summary>🤔 Question Socratique : Un modèle a un score de 0.95 sur train et 0.70 sur test. Que se passe-t-il ?</summary>

### 🔑 Réponse

C'est un cas classique d'**overfitting** (surapprentissage) !

- Le modèle performe excellemment sur les données qu'il a vues (train)
- Mais il performe beaucoup moins bien sur les nouvelles données (test)
- Il a **mémorisé** les exemples d'entraînement au lieu d'apprendre des patterns généraux

**Solutions possibles :**
- Simplifier le modèle (moins de paramètres)
- Ajouter de la régularisation
- Collecter plus de données d'entraînement

Nous approfondirons ce sujet au Chapitre 4 !

</details>

---

## 🔑 Paramètres vs Hyperparamètres

C'est une distinction **fondamentale** en Machine Learning :

```
┌─────────────────────────────────────────────────────────────────┐
│              PARAMÈTRES vs HYPERPARAMÈTRES                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   PARAMÈTRES                    HYPERPARAMÈTRES                 │
│   ══════════                    ════════════════                │
│                                                                 │
│   • Appris par le modèle        • Définis par VOUS              │
│   • Pendant .fit()              • AVANT .fit()                  │
│   • Automatiques                • Manuels (ou tuning)           │
│                                                                 │
│   Exemples :                    Exemples :                      │
│   - Coefficients (LinearReg)    - n_estimators (RandomForest)   │
│   - Poids des neurones (NN)     - max_depth (DecisionTree)      │
│   - Splits des arbres           - learning_rate (GradientBoost) │
│                                 - C (LogisticRegression)        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Exemple : Les hyperparamètres sont définis À LA CRÉATION du modèle
from sklearn.ensemble import RandomForestClassifier

# Ces valeurs sont des HYPERPARAMÈTRES (définis par nous)
rf = RandomForestClassifier(
    n_estimators=100,    # Nombre d'arbres
    max_depth=5,         # Profondeur maximale
    random_state=42
)

print("Hyperparamètres définis :")
print(f"  - n_estimators : {rf.n_estimators}")
print(f"  - max_depth : {rf.max_depth}")
print("\nCes valeurs ne changeront PAS pendant .fit()")

---

## 📖 Définition

```
┌─────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : API Scikit-Learn                                │
│                                                                 │
│ L'API (Application Programming Interface) de scikit-learn      │
│ définit une interface commune pour tous les modèles :          │
│                                                                 │
│ • .fit(X, y) — Entraîne le modèle sur les données              │
│ • .predict(X) — Génère des prédictions                         │
│ • .score(X, y) — Calcule la performance                        │
│                                                                 │
│ Cette uniformité permet de changer facilement de modèle        │
│ sans modifier le reste du code : un RandomForest se            │
│ substitue à une LinearRegression en une seule ligne.           │
└─────────────────────────────────────────────────────────────────┘
```

---

## 3.5 `.transform()` et `.fit_transform()` : Pour le preprocessing

Les **transformateurs** (scalers, encoders) utilisent un pattern similaire mais avec `.transform()` au lieu de `.predict()`.

| Objet | Méthodes principales |
|-------|---------------------|
| Modèle (estimator) | `.fit()`, `.predict()`, `.score()` |
| Transformateur | `.fit()`, `.transform()`, `.fit_transform()` |

In [ ]:
from sklearn.preprocessing import StandardScaler

# Créer le scaler
scaler = StandardScaler()

# FIT : apprend la moyenne et l'écart-type du TRAIN
scaler.fit(X_train)

print("Ce que le scaler a appris :")
print(f"  - Moyennes par feature : {scaler.mean_}")
print(f"  - Écarts-types par feature : {scaler.scale_}")

In [ ]:
# TRANSFORM : applique la transformation
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Utilise mean/std du TRAIN !

print("Avant scaling (train, 3 premiers exemples) :")
print(X_train[:3])
print("\nAprès scaling :")
print(X_train_scaled[:3])

### Le raccourci : `.fit_transform()`

Pour le train set, on fait souvent `.fit()` puis `.transform()`. Le raccourci `.fit_transform()` combine les deux :

In [ ]:
# Méthode longue (équivalente)
scaler1 = StandardScaler()
scaler1.fit(X_train)
X_train_v1 = scaler1.transform(X_train)

# Méthode courte (raccourci)
scaler2 = StandardScaler()
X_train_v2 = scaler2.fit_transform(X_train)  # fit + transform en une ligne

# Vérifions que c'est identique
print(f"Résultats identiques ? {np.allclose(X_train_v1, X_train_v2)}")

⚠️ **Attention !** N'utilisez **jamais** `.fit_transform()` sur le test set !

```python
# ✅ CORRECT
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)         # SEULEMENT transform

# ❌ ERREUR (data leakage !)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)     # fit sur test = LEAKAGE
```

<details>
<summary>🤔 Question Socratique : Pourquoi .fit_transform(X_test) est-il une erreur ?</summary>

### 🔑 Réponse

`.fit_transform(X_test)` calculerait la moyenne et l'écart-type **à partir du test set**, ce qui :

1. **Introduit du data leakage** — Le preprocessing utilise des informations du test
2. **N'est pas reproductible en production** — En production, vous n'aurez que de nouvelles données, pas un "test set" avec des statistiques connues
3. **Fausse l'évaluation** — Train et test sont normalisés différemment

**Règle :** `.fit_transform()` UNIQUEMENT sur train, `.transform()` sur test et production.

</details>

---

## 🎯 Exercice pratique : Le pattern complet

Mettons tout ensemble avec un exemple de classification.

In [ ]:
# Charger le dataset Iris (classification)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"Dataset Iris : {X_iris.shape[0]} exemples, {X_iris.shape[1]} features")
print(f"Classes : {iris.target_names}")

In [ ]:
# ÉTAPE 1 : Séparer les données (avec stratification car classification)
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris,
    test_size=0.2,
    stratify=y_iris,     # Préserver les proportions de classes
    random_state=42
)

print(f"Train : {len(X_train_iris)} exemples")
print(f"Test : {len(X_test_iris)} exemples")

In [ ]:
# ÉTAPE 2 : Créer et entraîner le modèle
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train_iris, y_train_iris)

print("✅ Modèle entraîné !")

In [ ]:
# ÉTAPE 3 : Prédire
y_pred_iris = clf.predict(X_test_iris)

print("Premières prédictions :")
for i in range(5):
    pred_name = iris.target_names[y_pred_iris[i]]
    true_name = iris.target_names[y_test_iris[i]]
    match = "✓" if pred_name == true_name else "✗"
    print(f"  {match} Prédit: {pred_name:12} | Réel: {true_name}")

In [ ]:
# ÉTAPE 4 : Évaluer
score_train_iris = clf.score(X_train_iris, y_train_iris)
score_test_iris = clf.score(X_test_iris, y_test_iris)

print(f"\n📊 Performance :")
print(f"  - Accuracy train : {score_train_iris:.2%}")
print(f"  - Accuracy test  : {score_test_iris:.2%}")

# Analyse de l'overfitting
ecart = score_train_iris - score_test_iris
if ecart < 0.05:
    print(f"\n✅ Pas d'overfitting détecté (écart de {ecart:.2%})")
else:
    print(f"\n⚠️ Possible overfitting (écart de {ecart:.2%})")

---

## 🔄 L'interchangeabilité des modèles

Grâce à l'API uniforme, changer de modèle est trivial :

In [ ]:
# Testons plusieurs modèles avec LE MÊME CODE
modeles = [
    ("Decision Tree", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=50, random_state=42)),
    ("Logistic Regression", LogisticRegression(max_iter=200, random_state=42)),
]

print("Comparaison des modèles :")
print("─" * 50)

for nom, modele in modeles:
    # Même pattern pour TOUS les modèles !
    modele.fit(X_train_iris, y_train_iris)
    score = modele.score(X_test_iris, y_test_iris)
    print(f"{nom:25} → Accuracy: {score:.2%}")

**C'est la puissance de l'API uniforme** : le code reste identique, seul le modèle change !

---

## 🧠 Réflexion métacognitive

1. **Pouvez-vous décrire le pattern fit/predict de mémoire ?** Quelles sont les 3 méthodes principales ?

2. **Quelle est la différence entre paramètres et hyperparamètres ?** Donnez un exemple de chaque.

3. **Pourquoi l'API uniforme de scikit-learn est-elle si pratique ?**

---

## 📝 Résumé

| Méthode | Rôle | Utilisé pour |
|---------|------|-------------|
| `.fit(X, y)` | Entraîner | Apprendre les paramètres |
| `.predict(X)` | Prédire | Générer des prédictions |
| `.score(X, y)` | Évaluer | Calculer la performance |
| `.transform(X)` | Transformer | Preprocessing (scalers, encoders) |
| `.fit_transform(X)` | Fit + Transform | Raccourci pour le train set uniquement |

**Règles d'or :**
- Toujours `.fit()` AVANT `.predict()` ou `.transform()`
- Ne jamais `.fit()` sur le test set
- Comparer train score et test score pour détecter l'overfitting

---

## ➡️ Prochaine leçon

Dans la **Leçon 2.4 : Pipelines scikit-learn**, nous allons découvrir comment **enchaîner** preprocessing et modèle en une seule structure — propre, reproductible, et sans risque de data leakage.

**Question de transition :** Comment garantir que le preprocessing et l'entraînement utilisent toujours les mêmes données, dans le bon ordre ?